In [4]:
"""
================================================================================
EKSPERIMEN FAKTORIAL 2x2 — Dekomposisi Kontribusi Inisialisasi dan Fitur
================================================================================
Untuk Subbab IV.11.1, Disertasi "Pengklasteran Citra Berbasis Data Simbolik
Menggunakan Algoritma K-Means".

Konfigurasi yang dijalankan:

    | Kode | Representasi Fitur          | Inisialisasi              |
    |------|-----------------------------|---------------------------|
    | A    | Simbolik (17 dim)           | Acak, 30 seed             |
    | B    | Simbolik (17 dim)           | Cartesian Join (seeded)   |
    | C    | ECDF/CDF per kanal (3x10)   | Acak, 30 seed             |
    | D    | ECDF/CDF per kanal (3x10)   | Cartesian Join (seeded)   |

Metrik yang dilaporkan: Akurasi (Hungarian), Akurasi (identitas, khusus seeded),
Purity terboboti, ARI, NMI, jumlah iterasi, waktu komputasi.

Cara pakai:
    1. Sesuaikan blok KONFIGURASI di bawah.
    2. python eksperimen_2x2.py
    3. Hasil tersimpan di hasil_2x2_ringkasan.csv dan hasil_2x2_mentah.csv,
       plus tabel siap-tempel di hasil_2x2_tabel.md

Kebutuhan: numpy, pillow, opencv-python, scikit-learn, scipy, pandas
================================================================================
"""

import os
import time
import json
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from scipy.optimize import linear_sum_assignment
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# -----------------------------------------------------------------------------
# Deteksi tepi: OpenCV bila tersedia, scikit-image sebagai cadangan.
#
# PENTING: gunakan OpenCV bila memungkinkan. Implementasi Canny pada
# scikit-image berbeda (thresholding pada magnitudo gradien ternormalisasi,
# bukan pada skala 0-255), sehingga fitur spasial yang dihasilkan tidak
# identik dan konfigurasi B tidak akan mereproduksi angka pada Tabel 46
# secara persis. Cadangan ini hanya untuk memastikan skrip tetap berjalan.
#
#     pip install opencv-python
# -----------------------------------------------------------------------------
EDGE_BACKEND = None

try:
    import cv2
    EDGE_BACKEND = "opencv"
except ImportError:
    try:
        from skimage.feature import canny as _sk_canny
        EDGE_BACKEND = "skimage"
        print("PERINGATAN: opencv-python tidak ditemukan, memakai scikit-image.")
        print("            Hasil deteksi tepi TIDAK identik dengan kode asli Anda.")
        print("            Untuk reproduksi persis: pip install opencv-python")
    except ImportError:
        raise ImportError(
            "Perlu opencv-python atau scikit-image untuk deteksi tepi Canny.\n"
            "Jalankan salah satu:\n"
            "    pip install opencv-python      (disarankan)\n"
            "    pip install scikit-image"
        )


def _ke_grayscale(image_data):
    """Grayscale dengan bobot luma ITU-R BT.601, sama seperti cv2.COLOR_RGB2GRAY."""
    if EDGE_BACKEND == "opencv":
        return cv2.cvtColor(image_data, cv2.COLOR_RGB2GRAY)
    return (0.299 * image_data[:, :, 0]
            + 0.587 * image_data[:, :, 1]
            + 0.114 * image_data[:, :, 2]).astype(np.uint8)


def _deteksi_tepi(gray, ambang_bawah=50, ambang_atas=150):
    """Deteksi tepi Canny, mengembalikan citra biner."""
    if EDGE_BACKEND == "opencv":
        return cv2.Canny(gray, ambang_bawah, ambang_atas, apertureSize=3)
    return _sk_canny(gray.astype(float) / 255.0, sigma=1.0,
                     low_threshold=ambang_bawah / 255.0,
                     high_threshold=ambang_atas / 255.0)


# =============================================================================
# KONFIGURASI — HANYA BAGIAN INI YANG PERLU ANDA UBAH
# =============================================================================

# -----------------------------------------------------------------------------
# SUMBER DATA
#   "download" : unduh CIFAR-10 resmi (10.000 citra uji, label terjamin benar).
#                Tidak perlu folder citra sama sekali. DISARANKAN.
#   "folder"   : baca dari folder citra di disk (mode lama).
# -----------------------------------------------------------------------------
DATA_SOURCE = "download"

DATA_ROOT = "./data_cifar10"     # tempat unduhan disimpan

# Jumlah citra latih per kelas untuk membentuk sentroid ter-supervisi.
# None = seluruh 5.000 citra per kelas (lebih lambat, lebih representatif).
N_TRAIN_PER_CLASS = 1000

# --- Berikut hanya dipakai bila DATA_SOURCE = "folder" ---
BASE = "C:/Users/Ahsan/Downloads/Colab Notebooks/Cifar 10"

TRAIN_DIR = os.path.join(BASE, "train")        # berisi subfolder per kelas
TEST_DIR = os.path.join(BASE, "test cifar")    # citra uji

# Urutan kelas. Pada mode "download" diganti otomatis dengan urutan resmi
# CIFAR-10 agar indeks klaster konsisten dengan label bawaan dataset.
CLASS_NAMES = ["truck", "airplane", "automobile", "bird", "cat",
               "deer", "dog", "frog", "horse", "ship"]

# Bagaimana label kebenaran citra uji ditentukan.
#   "subfolder" : TEST_DIR berisi subfolder per kelas (test/truck/xxx.png)
#   "filename"  : nama berkas diawali nama kelas (truck_0001.png)
#   "csv"       : ada berkas CSV dengan kolom "filename" dan "label"
TEST_LABEL_MODE = "subfolder"
TEST_LABEL_CSV = os.path.join(BASE, "test_labels.csv")

IMG_EXT = (".png", ".jpg", ".jpeg")

N_CLUSTERS = 10
N_SEEDS = 30                # jumlah ulangan untuk konfigurasi A dan C
BINS = 10                   # jumlah bin ECDF per kanal -> fitur 3*10 = 30 dim
MAX_ITER = 300              # JANGAN 10 (lihat butir 3.3 naskah perbaikan)
TOL = 1e-6

# Standardisasi fitur simbolik sebelum K-Means Euclidean.
# False = mereproduksi kode asli Anda (skala RGB 0-255 mendominasi posisi 0-32).
# True  = metodologis lebih benar. Jalankan keduanya, laporkan yang False
#         sebagai reproduksi dan yang True sebagai analisis sensitivitas.
STANDARDIZE_SYMBOLIC = False

CACHE_DIR = "./cache_fitur"
OUT_PREFIX = "hasil_2x2"

RANDOM_INIT_METHOD = "k-means++"   # atau "random" (Forgy)

# Cara membentuk sentroid ter-supervisi dari citra latih satu kelas.
#   "mean"     : rata-rata fitur (seperti kode asli Anda)
#   "interval" : min dari min, max dari max (definisi Cartesian Join, II.13)
AGREGASI_SENTROID = "mean"

np.seterr(divide="ignore", invalid="ignore")
warnings.filterwarnings("ignore", category=RuntimeWarning)


# =============================================================================
# 1. EKSTRAKSI FITUR
# =============================================================================

def _load_rgb(path):
    """Baca citra sebagai array RGB uint8 (n_baris, n_kolom, 3)."""
    img = Image.open(path).convert("RGB")
    return np.array(img)


def fitur_simbolik(image_data):
    """
    Fitur simbolik 17 dimensi — identik dengan get_image_features() milik Anda.
    Urutan dipertahankan persis agar hasil konfigurasi B dapat direproduksi.
    """
    avg_rgb = image_data.mean(axis=(0, 1))
    min_rgb = image_data.min(axis=(0, 1))
    max_rgb = image_data.max(axis=(0, 1))

    gray = _ke_grayscale(image_data)
    edges = _deteksi_tepi(gray, 50, 150)
    y_pos, x_pos = np.where(edges != 0)

    if len(x_pos) > 0:
        avg_x, avg_y = float(np.mean(x_pos)), float(np.mean(y_pos))
        min_x, max_x = float(np.min(x_pos)), float(np.max(x_pos))
        min_y, max_y = float(np.min(y_pos)), float(np.max(y_pos))
        length_x, length_y = max_x - min_x, max_y - min_y
    else:
        avg_x = avg_y = min_x = max_x = min_y = max_y = 0.0
        length_x = length_y = 0.0

    return np.array([
        avg_rgb[0], avg_rgb[1], avg_rgb[2],
        avg_x, avg_y,
        min_rgb[0], min_rgb[1], min_rgb[2],
        max_rgb[0], max_rgb[1], max_rgb[2],
        length_x, length_y,
        min_x, max_x, min_y, max_y,
    ], dtype=np.float64)


def fitur_ecdf(image_data, bins=BINS):
    """
    Fitur distribusi per kanal warna, bentuk (3, bins).

    Mengembalikan PMF (frekuensi relatif) karena divergensi Jensen-Shannon
    didefinisikan atas distribusi peluang. CDF diperoleh dengan np.cumsum
    dan dipakai untuk statistik Kolmogorov-Smirnov bila diperlukan.
    """
    out = np.zeros((3, bins), dtype=np.float64)
    for ch in range(3):
        hist, _ = np.histogram(image_data[:, :, ch], bins=bins, range=(0, 256))
        total = hist.sum()
        out[ch] = hist / total if total > 0 else np.full(bins, 1.0 / bins)
    return out


def daftar_citra(folder):
    return sorted(
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(IMG_EXT)
    )


def ekstrak_folder(paths, desc=""):
    """Ekstrak kedua jenis fitur sekaligus agar citra hanya dibaca satu kali."""
    n = len(paths)
    sym = np.zeros((n, 17), dtype=np.float64)
    ecdf = np.zeros((n, 3, BINS), dtype=np.float64)
    for i, p in enumerate(paths):
        img = _load_rgb(p)
        sym[i] = fitur_simbolik(img)
        ecdf[i] = fitur_ecdf(img)
        if desc and (i + 1) % 500 == 0:
            print(f"  {desc}: {i + 1}/{n}", flush=True)
    return sym, ecdf


def ekstrak_array(images, desc=""):
    """Ekstrak kedua jenis fitur dari array citra (n, tinggi, lebar, 3) uint8."""
    n = len(images)
    sym = np.zeros((n, 17), dtype=np.float64)
    ecdf = np.zeros((n, 3, BINS), dtype=np.float64)
    for i in range(n):
        img = np.ascontiguousarray(images[i], dtype=np.uint8)
        sym[i] = fitur_simbolik(img)
        ecdf[i] = fitur_ecdf(img)
        if desc and (i + 1) % 1000 == 0:
            print(f"  {desc}: {i + 1}/{n}", flush=True)
    return sym, ecdf


# =============================================================================
# 2. PEMUATAN DATA + LABEL KEBENARAN
# =============================================================================

def unduh_cifar10():
    """
    Unduh CIFAR-10 resmi. Mengembalikan (x_train, y_train, x_test, y_test,
    nama_kelas) dengan citra uint8 RGB berbentuk (n, 32, 32, 3).

    Mencoba torchvision lebih dulu, lalu tensorflow.keras. Keduanya mengunduh
    dari sumber resmi (Krizhevsky, University of Toronto), sehingga label
    kebenarannya terjamin dan hasilnya dapat direproduksi siapa pun tanpa
    bergantung pada struktur folder lokal.
    """
    try:
        from torchvision.datasets import CIFAR10
        tr = CIFAR10(root=DATA_ROOT, train=True, download=True)
        te = CIFAR10(root=DATA_ROOT, train=False, download=True)
        print("Sumber data: torchvision.datasets.CIFAR10")
        return (np.asarray(tr.data), np.asarray(tr.targets),
                np.asarray(te.data), np.asarray(te.targets), list(tr.classes))
    except ImportError:
        pass

    try:
        from tensorflow.keras.datasets import cifar10
        (xtr, ytr), (xte, yte) = cifar10.load_data()
        print("Sumber data: tensorflow.keras.datasets.cifar10")
        nama = ["airplane", "automobile", "bird", "cat", "deer",
                "dog", "frog", "horse", "ship", "truck"]
        return xtr, ytr.ravel(), xte, yte.ravel(), nama
    except ImportError:
        pass

    raise ImportError(
        "Perlu torchvision atau tensorflow untuk mengunduh CIFAR-10.\n"
        "Jalankan salah satu:\n"
        "    pip install torchvision      (jauh lebih ringan, disarankan)\n"
        "    pip install tensorflow\n"
        "Atau ubah DATA_SOURCE = 'folder' bila datanya sudah ada di disk."
    )


def muat_dari_unduhan():
    """Siapkan fitur uji dan sentroid ter-supervisi dari CIFAR-10 resmi."""
    global CLASS_NAMES

    os.makedirs(CACHE_DIR, exist_ok=True)
    cache = os.path.join(
        CACHE_DIR, f"unduh_{BINS}_{N_TRAIN_PER_CLASS}_{EDGE_BACKEND}.npz")

    if os.path.exists(cache):
        print(f"Memuat fitur dari cache: {cache}")
        d = np.load(cache, allow_pickle=True)
        CLASS_NAMES = list(d["kelas"])
        return (d["y"], d["sym"], d["ecdf"], d["c_sym"], d["c_ecdf"])

    xtr, ytr, xte, yte, nama = unduh_cifar10()
    CLASS_NAMES = nama
    print(f"Urutan kelas resmi CIFAR-10: {CLASS_NAMES}")
    print(f"Himpunan uji: {len(xte)} citra")
    print("Sebaran per kelas:",
          {CLASS_NAMES[i]: int((yte == i).sum()) for i in range(len(CLASS_NAMES))})

    print("\nEkstraksi fitur himpunan uji...")
    sym, ecdf = ekstrak_array(xte, desc="uji")

    print("\nMembentuk sentroid ter-supervisi dari himpunan latih...")
    c_sym = np.zeros((N_CLUSTERS, 17))
    c_ecdf = np.zeros((N_CLUSTERS, 3, BINS))

    for ci, cname in enumerate(CLASS_NAMES):
        idx = np.where(ytr == ci)[0]
        if N_TRAIN_PER_CLASS is not None:
            idx = idx[:N_TRAIN_PER_CLASS]
        s, e = ekstrak_array(xtr[idx])
        c_sym[ci] = agregasi_sentroid_simbolik(s)
        m = e.mean(axis=0)
        c_ecdf[ci] = m / m.sum(axis=1, keepdims=True)
        print(f"  sentroid '{cname}' dari {len(idx)} citra latih")

    np.savez_compressed(cache, y=yte, sym=sym, ecdf=ecdf,
                        c_sym=c_sym, c_ecdf=c_ecdf,
                        kelas=np.array(CLASS_NAMES))
    return yte, sym, ecdf, c_sym, c_ecdf


def agregasi_sentroid_simbolik(sym):
    """
    Bentuk satu sentroid simbolik dari kumpulan fitur satu kelas.

    CATATAN PENTING — periksa definisi operasi ini.
    Kode asli Anda memakai mean per kelas. Definisi Cartesian Join pada
    Subbab II.13 adalah operasi interval (min dari min, max dari max).
    Bila naskah menyebut Cartesian Join tetapi kode memakai mean, salah satu
    dari keduanya harus diperbaiki agar konsisten.
    """
    if AGREGASI_SENTROID == "mean":
        return sym.mean(axis=0)

    v = np.empty(17)
    v[[0, 1, 2, 3, 4]] = sym[:, [0, 1, 2, 3, 4]].mean(axis=0)
    v[[5, 6, 7]] = sym[:, [5, 6, 7]].min(axis=0)      # min RGB
    v[[8, 9, 10]] = sym[:, [8, 9, 10]].max(axis=0)    # max RGB
    v[[11, 12]] = sym[:, [11, 12]].mean(axis=0)       # length
    v[[13, 15]] = sym[:, [13, 15]].min(axis=0)        # min x, min y
    v[[14, 16]] = sym[:, [14, 16]].max(axis=0)        # max x, max y
    return v

def muat_data_uji():
    """Kembalikan (paths, y_true, sym, ecdf) untuk himpunan uji, dengan cache."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    cache = os.path.join(
        CACHE_DIR, f"uji_{TEST_LABEL_MODE}_{BINS}_{EDGE_BACKEND}.npz")

    if os.path.exists(cache):
        print(f"Memuat fitur uji dari cache: {cache}")
        d = np.load(cache, allow_pickle=True)
        return list(d["paths"]), d["y"], d["sym"], d["ecdf"]

    paths, y = [], []

    if TEST_LABEL_MODE == "subfolder":
        for ci, cname in enumerate(CLASS_NAMES):
            sub = os.path.join(TEST_DIR, cname)
            if not os.path.isdir(sub):
                raise FileNotFoundError(
                    f"Subfolder kelas tidak ditemukan: {sub}\n"
                    f"Ubah TEST_LABEL_MODE bila struktur foldernya berbeda."
                )
            for p in daftar_citra(sub):
                paths.append(p)
                y.append(ci)

    elif TEST_LABEL_MODE == "filename":
        idx = {c: i for i, c in enumerate(CLASS_NAMES)}
        for p in daftar_citra(TEST_DIR):
            base = os.path.basename(p).lower()
            cocok = [c for c in CLASS_NAMES if base.startswith(c.lower())]
            if not cocok:
                raise ValueError(
                    f"Kelas tidak terbaca dari nama berkas: {base}\n"
                    f"Sesuaikan logika parsing atau pakai TEST_LABEL_MODE lain."
                )
            paths.append(p)
            y.append(idx[max(cocok, key=len)])

    elif TEST_LABEL_MODE == "csv":
        df = pd.read_csv(TEST_LABEL_CSV)
        idx = {c: i for i, c in enumerate(CLASS_NAMES)}
        for _, r in df.iterrows():
            paths.append(os.path.join(TEST_DIR, str(r["filename"])))
            y.append(idx[str(r["label"])])

    else:
        raise ValueError(f"TEST_LABEL_MODE tidak dikenal: {TEST_LABEL_MODE}")

    y = np.asarray(y, dtype=int)
    print(f"Himpunan uji: {len(paths)} citra, {len(np.unique(y))} kelas")
    print("Sebaran per kelas:",
          {CLASS_NAMES[i]: int((y == i).sum()) for i in np.unique(y)})

    sym, ecdf = ekstrak_folder(paths, desc="uji")
    np.savez_compressed(cache, paths=np.array(paths), y=y, sym=sym, ecdf=ecdf)
    return paths, y, sym, ecdf


def bangun_sentroid_seeded():
    """
    Sentroid awal ter-supervisi (mode folder): satu sentroid per kelas,
    dibentuk dari citra latih yang labelnya diketahui. Inilah sumber kebocoran
    label pada konfigurasi B dan D, dan harus dinyatakan eksplisit di naskah.
    """
    os.makedirs(CACHE_DIR, exist_ok=True)
    cache = os.path.join(
        CACHE_DIR, f"seed_{AGREGASI_SENTROID}_{BINS}_{EDGE_BACKEND}.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        return d["sym"], d["ecdf"]

    c_sym = np.zeros((N_CLUSTERS, 17))
    c_ecdf = np.zeros((N_CLUSTERS, 3, BINS))

    for ci, cname in enumerate(CLASS_NAMES):
        folder = os.path.join(TRAIN_DIR, cname)
        paths = daftar_citra(folder)
        if not paths:
            raise FileNotFoundError(f"Tidak ada citra di {folder}")
        sym, ecdf = ekstrak_folder(paths, desc=f"latih/{cname}")

        c_sym[ci] = agregasi_sentroid_simbolik(sym)
        # Sentroid distribusi selalu rata-rata pointwise (Lemma III.4.2)
        m = ecdf.mean(axis=0)
        c_ecdf[ci] = m / m.sum(axis=1, keepdims=True)

        print(f"  sentroid '{cname}' dari {len(paths)} citra")

    np.savez_compressed(cache, sym=c_sym, ecdf=c_ecdf)
    return c_sym, c_ecdf


# =============================================================================
# 3. K-MEANS DENGAN DIVERGENSI JENSEN-SHANNON
# =============================================================================

EPS = 1e-12


def _entropi(p):
    """Entropi Shannon basis 2, dijumlahkan pada sumbu terakhir."""
    return -np.sum(np.where(p > EPS, p * np.log2(np.maximum(p, EPS)), 0.0), axis=-1)


def js_divergence(P, Q):
    """
    JS(P, Q) basis 2, dirata-ratakan atas kanal warna.
    P: (n, 3, b) atau (3, b) — Q: (k, 3, b)
    Mengembalikan matriks (n, k) berisi nilai pada rentang [0, 1].
    """
    P = np.atleast_3d(P) if P.ndim == 3 else P[None, ...]
    a = P[:, None, :, :]          # (n, 1, 3, b)
    b = Q[None, :, :, :]          # (1, k, 3, b)
    M = 0.5 * (a + b)
    js = _entropi(M) - 0.5 * _entropi(a) - 0.5 * _entropi(b)   # (n, k, 3)
    return np.clip(js.mean(axis=2), 0.0, 1.0)


class JSKMeans:
    """
    K-Means pada ruang distribusi dengan divergensi Jensen-Shannon.

    Pembaruan sentroid memakai rata-rata pointwise dengan PENGAMAN MONOTON
    (monotone safeguard): sentroid hanya diperbarui bila tidak menaikkan
    fungsi objektif. Ini menjamin syarat monotonisitas Teorema III.4.1 tanpa
    perlu mengklaim rata-rata pointwise sebagai minimizer global JS — klaim
    yang tidak berlaku karena JS bukan Bregman divergence.

    Atribut n_safeguard mencatat berapa kali pengaman aktif. Bila nilainya 0,
    itu bukti empiris bahwa rata-rata pointwise merupakan aproksimasi yang
    sangat baik untuk data Anda, dan layak dilaporkan di Subbab III.4.4.
    """

    def __init__(self, n_clusters=10, init=None, max_iter=MAX_ITER,
                 tol=TOL, random_state=None, init_method=RANDOM_INIT_METHOD):
        self.n_clusters = n_clusters
        self.init = init
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.init_method = init_method

    def _init_centroids(self, X, rng):
        if self.init is not None:
            return self.init.copy()

        n = X.shape[0]
        if self.init_method == "random":
            idx = rng.choice(n, self.n_clusters, replace=False)
            return X[idx].copy()

        # k-means++ dengan JS sebagai ukuran jarak
        centers = [X[rng.integers(n)]]
        for _ in range(1, self.n_clusters):
            d = js_divergence(X, np.array(centers)).min(axis=1) ** 2
            total = d.sum()
            prob = np.full(n, 1.0 / n) if total <= 0 else d / total
            centers.append(X[rng.choice(n, p=prob)])
        return np.array(centers)

    def _objective(self, X, C, labels):
        d = js_divergence(X, C)
        return float(d[np.arange(len(X)), labels].sum())

    def fit(self, X):
        rng = np.random.default_rng(self.random_state)
        C = self._init_centroids(X, rng)

        self.n_safeguard = 0
        prev_obj = np.inf
        labels = np.zeros(len(X), dtype=int)

        for it in range(1, self.max_iter + 1):
            # --- Langkah penugasan ---
            labels = js_divergence(X, C).argmin(axis=1)
            obj = self._objective(X, C, labels)

            # --- Langkah pembaruan dengan pengaman monoton ---
            C_baru = C.copy()
            for k in range(self.n_clusters):
                anggota = X[labels == k]
                if len(anggota) == 0:
                    continue
                kandidat = anggota.mean(axis=0)
                kandidat = kandidat / np.maximum(
                    kandidat.sum(axis=1, keepdims=True), EPS)

                j_lama = js_divergence(anggota, C[k][None]).sum()
                j_baru = js_divergence(anggota, kandidat[None]).sum()
                if j_baru <= j_lama:
                    C_baru[k] = kandidat
                else:
                    self.n_safeguard += 1

            C = C_baru
            if abs(prev_obj - obj) < self.tol:
                break
            prev_obj = obj

        self.cluster_centers_ = C
        self.labels_ = js_divergence(X, C).argmin(axis=1)
        self.n_iter_ = it
        self.inertia_ = self._objective(X, C, self.labels_)
        return self


# =============================================================================
# 4. METRIK EVALUASI
# =============================================================================

def akurasi_hungarian(y_true, y_pred, n_clusters=N_CLUSTERS):
    """
    Akurasi setelah pemetaan bijektif klaster->kelas yang optimal
    (Linear Assignment Problem, Munkres 1957). Dipakai untuk konfigurasi
    A dan C, yang inisialisasinya acak sehingga urutan klaster arbitrer.
    """
    n_kelas = max(len(np.unique(y_true)), n_clusters)
    M = np.zeros((n_clusters, n_kelas), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        M[p, t] += 1
    baris, kolom = linear_sum_assignment(-M)
    pemetaan = {int(b): int(k) for b, k in zip(baris, kolom)}
    benar = M[baris, kolom].sum()
    return float(benar) / len(y_true), pemetaan


def akurasi_identitas(y_true, y_pred):
    """
    Akurasi tanpa pemetaan: klaster ke-k dianggap kelas ke-k.
    Sahih HANYA untuk konfigurasi seeded (B dan D), karena di sana
    korespondensi klaster-kelas sudah ditetapkan pada tahap inisialisasi.
    """
    return float(np.mean(np.asarray(y_pred) == np.asarray(y_true)))


def purity_per_klaster(y_true, y_pred, n_clusters=N_CLUSTERS):
    """
    Purity tiap klaster — metrik yang menghasilkan angka 82-91% pada Tabel 46.
    Berbeda dari akurasi keseluruhan; disertakan agar Tabel 46 dapat
    direproduksi dan perbedaannya dapat dijelaskan di naskah.
    """
    baris = []
    total_benar = 0
    for k in range(n_clusters):
        mask = (y_pred == k)
        N = int(mask.sum())
        if N == 0:
            baris.append({"klaster": k, "kelas_mayoritas": None,
                          "error": 0, "N": 0, "purity_persen": np.nan})
            continue
        cnt = np.bincount(y_true[mask], minlength=len(CLASS_NAMES))
        mayor = int(cnt.argmax())
        benar = int(cnt[mayor])
        total_benar += benar
        baris.append({
            "klaster": k,
            "kelas_mayoritas": CLASS_NAMES[mayor],
            "error": N - benar,
            "N": N,
            "purity_persen": round(100.0 * benar / N, 2),
        })
    return pd.DataFrame(baris), float(total_benar) / len(y_true)


def evaluasi(y_true, y_pred, seeded):
    acc_h, pemetaan = akurasi_hungarian(y_true, y_pred)
    tabel, purity = purity_per_klaster(y_true, y_pred)
    return {
        "akurasi_hungarian": acc_h,
        "akurasi_identitas": akurasi_identitas(y_true, y_pred) if seeded else np.nan,
        "purity_terboboti": purity,
        "ari": adjusted_rand_score(y_true, y_pred),
        "nmi": normalized_mutual_info_score(y_true, y_pred),
    }, tabel, pemetaan


# =============================================================================
# 5. PELAKSANAAN TIAP KONFIGURASI
# =============================================================================

def jalankan_simbolik(X, y, init=None, seed=None):
    t0 = time.time()
    if init is not None:
        km = KMeans(n_clusters=N_CLUSTERS, init=init, n_init=1,
                    max_iter=MAX_ITER, tol=TOL, random_state=0)
    else:
        km = KMeans(n_clusters=N_CLUSTERS, init=RANDOM_INIT_METHOD, n_init=1,
                    max_iter=MAX_ITER, tol=TOL, random_state=seed)
    km.fit(X)
    lama = time.time() - t0

    hasil, tabel, _ = evaluasi(y, km.labels_, seeded=init is not None)
    hasil.update(iterasi=int(km.n_iter_), waktu_detik=lama, safeguard=0)
    return hasil, tabel


def jalankan_ecdf(X, y, init=None, seed=None):
    t0 = time.time()
    km = JSKMeans(n_clusters=N_CLUSTERS, init=init,
                  max_iter=MAX_ITER, tol=TOL, random_state=seed)
    km.fit(X)
    lama = time.time() - t0

    hasil, tabel, _ = evaluasi(y, km.labels_, seeded=init is not None)
    hasil.update(iterasi=int(km.n_iter_), waktu_detik=lama,
                 safeguard=int(km.n_safeguard))
    return hasil, tabel


# =============================================================================
# 6. PROGRAM UTAMA
# =============================================================================

def periksa_direktori():
    """
    Pemeriksaan awal agar kesalahan path terdeteksi dalam hitungan detik,
    bukan setelah ekstraksi fitur berjalan setengah jalan.
    """
    masalah = []

    for nama, d in [("TRAIN_DIR", TRAIN_DIR), ("TEST_DIR", TEST_DIR)]:
        if not os.path.isdir(d):
            masalah.append(f"{nama} tidak ditemukan: {d}")
            induk = os.path.dirname(d.rstrip("/\\"))
            if os.path.isdir(induk):
                isi = sorted(os.listdir(induk))[:15]
                masalah.append(f"    isi dari {induk}: {isi}")

    if os.path.isdir(TRAIN_DIR):
        hilang = [c for c in CLASS_NAMES
                  if not os.path.isdir(os.path.join(TRAIN_DIR, c))]
        if hilang:
            masalah.append(f"Subfolder kelas tidak ada di TRAIN_DIR: {hilang}")
            masalah.append(f"    yang tersedia: {sorted(os.listdir(TRAIN_DIR))[:15]}")

    if TEST_LABEL_MODE == "subfolder" and os.path.isdir(TEST_DIR):
        hilang = [c for c in CLASS_NAMES
                  if not os.path.isdir(os.path.join(TEST_DIR, c))]
        if hilang:
            masalah.append(
                f"TEST_LABEL_MODE='subfolder' tetapi subfolder kelas tidak ada: {hilang}")
            masalah.append(f"    isi TEST_DIR: {sorted(os.listdir(TEST_DIR))[:15]}")
            masalah.append("    Bila folder uji datar, ganti TEST_LABEL_MODE "
                           "menjadi 'filename' atau 'csv'.")

    if masalah:
        raise FileNotFoundError(
            "Pemeriksaan direktori gagal:\n  " + "\n  ".join(masalah))

    print("Pemeriksaan direktori: OK")


def main():
    print("=" * 78)
    print("EKSPERIMEN FAKTORIAL 2x2 — Inisialisasi x Representasi Fitur")
    print("=" * 78)
    print(f"Backend deteksi tepi : {EDGE_BACKEND}")
    print(f"Sumber data          : {DATA_SOURCE}")
    print(f"Agregasi sentroid    : {AGREGASI_SENTROID}")
    print()

    if DATA_SOURCE == "download":
        y, sym, ecdf, c_sym, c_ecdf = muat_dari_unduhan()
    elif DATA_SOURCE == "folder":
        print(f"Direktori latih : {TRAIN_DIR}")
        print(f"Direktori uji   : {TEST_DIR}\n")
        periksa_direktori()
        _, y, sym, ecdf = muat_data_uji()
        print("\nMembangun sentroid ter-supervisi dari folder latih...")
        c_sym, c_ecdf = bangun_sentroid_seeded()
    else:
        raise ValueError(f"DATA_SOURCE tidak dikenal: {DATA_SOURCE}")

    if STANDARDIZE_SYMBOLIC:
        mu, sd = sym.mean(axis=0), sym.std(axis=0)
        sym_pakai = (sym - mu) / np.where(sd > 0, sd, 1.0)
        c_sym = (c_sym - mu) / np.where(sd > 0, sd, 1.0)
        print("\nFitur simbolik distandardisasi (z-score).")
    else:
        sym_pakai = sym
        print("\nFitur simbolik TIDAK distandardisasi (mereproduksi kode asli).")

    baris = []
    tabel_purity = {}

    # ---- B: simbolik + seeded (deterministik, 1 kali) ----
    print("\n[B] Simbolik + Cartesian Join ter-supervisi ...")
    h, t = jalankan_simbolik(sym_pakai, y, init=c_sym)
    h.update(konfigurasi="B", fitur="Simbolik", inisialisasi="Seeded", seed=-1)
    baris.append(h)
    tabel_purity["B"] = t
    print(f"    akurasi identitas = {h['akurasi_identitas']:.4f} | "
          f"Hungarian = {h['akurasi_hungarian']:.4f} | "
          f"purity = {h['purity_terboboti']:.4f}")

    # ---- D: ECDF + seeded (deterministik, 1 kali) ----
    print("\n[D] ECDF + Cartesian Join ter-supervisi ...")
    h, t = jalankan_ecdf(ecdf, y, init=c_ecdf)
    h.update(konfigurasi="D", fitur="ECDF", inisialisasi="Seeded", seed=-1)
    baris.append(h)
    tabel_purity["D"] = t
    print(f"    akurasi identitas = {h['akurasi_identitas']:.4f} | "
          f"Hungarian = {h['akurasi_hungarian']:.4f} | "
          f"safeguard aktif {h['safeguard']}x")

    # ---- A dan C: inisialisasi acak, 30 seed ----
    seeds = np.random.default_rng(20260812).integers(1, 10000, size=N_SEEDS)
    print(f"\nSeed yang dipakai ({N_SEEDS}): {list(map(int, seeds))}")
    with open(f"{OUT_PREFIX}_seeds.json", "w") as f:
        json.dump({"seeds": list(map(int, seeds))}, f, indent=2)

    print("\n[A] Simbolik + inisialisasi acak ...")
    for i, s in enumerate(seeds, 1):
        h, _ = jalankan_simbolik(sym_pakai, y, init=None, seed=int(s))
        h.update(konfigurasi="A", fitur="Simbolik",
                 inisialisasi="Acak", seed=int(s))
        baris.append(h)
        print(f"    {i:2d}/{N_SEEDS} seed={s:<5d} "
              f"akurasi={h['akurasi_hungarian']:.4f}", flush=True)

    print("\n[C] ECDF + inisialisasi acak ...")
    for i, s in enumerate(seeds, 1):
        h, _ = jalankan_ecdf(ecdf, y, init=None, seed=int(s))
        h.update(konfigurasi="C", fitur="ECDF",
                 inisialisasi="Acak", seed=int(s))
        baris.append(h)
        print(f"    {i:2d}/{N_SEEDS} seed={s:<5d} "
              f"akurasi={h['akurasi_hungarian']:.4f}", flush=True)

    # ---- Ringkasan ----
    df = pd.DataFrame(baris)
    df.to_csv(f"{OUT_PREFIX}_mentah.csv", index=False)

    def ringkas(g):
        acc = g["akurasi_identitas"] if g.name in ("B", "D") else g["akurasi_hungarian"]
        acc = g["akurasi_hungarian"] if acc.isna().all() else acc
        return pd.Series({
            "fitur": g["fitur"].iloc[0],
            "inisialisasi": g["inisialisasi"].iloc[0],
            "n": len(g),
            "akurasi_mean": 100 * acc.mean(),
            "akurasi_sd": 100 * acc.std(ddof=1) if len(g) > 1 else 0.0,
            "hungarian_mean": 100 * g["akurasi_hungarian"].mean(),
            "purity_mean": 100 * g["purity_terboboti"].mean(),
            "ari_mean": g["ari"].mean(),
            "ari_sd": g["ari"].std(ddof=1) if len(g) > 1 else 0.0,
            "nmi_mean": g["nmi"].mean(),
            "nmi_sd": g["nmi"].std(ddof=1) if len(g) > 1 else 0.0,
            "iterasi_mean": g["iterasi"].mean(),
            "waktu_mean": g["waktu_detik"].mean(),
        })

    ring = df.groupby("konfigurasi").apply(ringkas).reindex(["A", "B", "C", "D"])
    ring.to_csv(f"{OUT_PREFIX}_ringkasan.csv")

    print("\n" + "=" * 78)
    print("RINGKASAN")
    print("=" * 78)
    print(ring.round(4).to_string())

    # ---- Dekomposisi faktorial ----
    print("\n" + "-" * 78)
    print("DEKOMPOSISI")
    print("-" * 78)
    a, b = ring.loc["A", "akurasi_mean"], ring.loc["B", "akurasi_mean"]
    c, d = ring.loc["C", "akurasi_mean"], ring.loc["D", "akurasi_mean"]
    print(f"  B - A  = {b - a:+6.2f} poin  kontribusi seeding pada fitur simbolik")
    print(f"  D - C  = {d - c:+6.2f} poin  kontribusi seeding pada fitur ECDF")
    print(f"  A vs C = {a:.2f}% vs {c:.2f}%  perbandingan fitur yang ADIL (tanpa label)")
    print(f"  B vs D = {b:.2f}% vs {d:.2f}%  perbandingan fitur dengan seeding")

    # ---- Uji t: A vs C ----
    ha = df[df.konfigurasi == "A"].sort_values("seed")["akurasi_hungarian"].values
    hc = df[df.konfigurasi == "C"].sort_values("seed")["akurasi_hungarian"].values
    t_stat, p_val = stats.ttest_rel(ha, hc)
    selisih = ha - hc
    cohen_d = selisih.mean() / selisih.std(ddof=1) if selisih.std(ddof=1) > 0 else np.nan
    print(f"\n  Uji t berpasangan A vs C: t = {t_stat:.3f}, p = {p_val:.3e}, "
          f"Cohen's d = {cohen_d:.3f}")

    # ---- Tabel 46a siap tempel ----
    def sel(k, kol_m, kol_s, desimal=2, persen=True):
        m = ring.loc[k, kol_m]
        s = ring.loc[k, kol_s]
        if ring.loc[k, "n"] == 1:
            return f"{m:.{desimal}f}"
        return f"{m:.{desimal}f} ± {s:.{desimal}f}"

    with open(f"{OUT_PREFIX}_tabel.md", "w") as f:
        f.write("### Tabel 46a. Dekomposisi faktorial pengaruh inisialisasi "
                "dan representasi fitur (CIFAR-10)\n\n")
        f.write("| Konfigurasi | Representasi Fitur | Inisialisasi | "
                "Akurasi (%) | ARI | NMI |\n")
        f.write("|---|---|---|---|---|---|\n")
        label = {
            "A": ("Simbolik (avg, max, min, length)", "Acak (30 seed)"),
            "B": ("Simbolik (avg, max, min, length)", "*Cartesian Join* ter-supervisi"),
            "C": ("ECDF/CDF per kanal", "Acak (30 seed)"),
            "D": ("ECDF/CDF per kanal", "*Cartesian Join* ter-supervisi"),
        }
        for k in ["A", "B", "C", "D"]:
            fitur_l, init_l = label[k]
            f.write(f"| {k} | {fitur_l} | {init_l} | "
                    f"{sel(k, 'akurasi_mean', 'akurasi_sd')} | "
                    f"{sel(k, 'ari_mean', 'ari_sd', 3)} | "
                    f"{sel(k, 'nmi_mean', 'nmi_sd', 3)} |\n")
        f.write("\nAkurasi konfigurasi A dan C dihitung setelah pemetaan Hungarian; "
                "konfigurasi B dan D memakai korespondensi klaster-kelas yang "
                "ditetapkan pada tahap inisialisasi.\n")

    # ---- Tabel 46 (purity per klaster, konfigurasi B) ----
    tabel_purity["B"].to_csv(f"{OUT_PREFIX}_tabel46_purity_B.csv", index=False)
    print(f"\n  Reproduksi Tabel 46 (purity per klaster, konfigurasi B):")
    print(tabel_purity["B"].to_string(index=False))
    print(f"\n  Total N = {int(tabel_purity['B']['N'].sum())} "
          f"(bandingkan dengan 8.348 pada naskah)")

    print(f"\nBerkas keluaran:")
    for nm in ["_mentah.csv", "_ringkasan.csv", "_tabel.md",
               "_seeds.json", "_tabel46_purity_B.csv"]:
        print(f"  {OUT_PREFIX}{nm}")


if __name__ == "__main__":
    main()

EKSPERIMEN FAKTORIAL 2x2 — Inisialisasi x Representasi Fitur
Backend deteksi tepi : opencv
Sumber data          : download
Agregasi sentroid    : mean

Sumber data: tensorflow.keras.datasets.cifar10
Urutan kelas resmi CIFAR-10: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Himpunan uji: 10000 citra
Sebaran per kelas: {'airplane': 1000, 'automobile': 1000, 'bird': 1000, 'cat': 1000, 'deer': 1000, 'dog': 1000, 'frog': 1000, 'horse': 1000, 'ship': 1000, 'truck': 1000}

Ekstraksi fitur himpunan uji...
  uji: 1000/10000
  uji: 2000/10000
  uji: 3000/10000
  uji: 4000/10000
  uji: 5000/10000
  uji: 6000/10000
  uji: 7000/10000
  uji: 8000/10000
  uji: 9000/10000
  uji: 10000/10000

Membentuk sentroid ter-supervisi dari himpunan latih...
  sentroid 'airplane' dari 1000 citra latih
  sentroid 'automobile' dari 1000 citra latih
  sentroid 'bird' dari 1000 citra latih
  sentroid 'cat' dari 1000 citra latih
  sentroid 'deer' dari 1000 citra latih
  sen